# Step 7: Quantum Phase Transition and Finite-Size Scaling in the TFIM

In notebook 02 we applied finite-size scaling (FSS) to the classical Ising transition: we located $T_c$, measured the Binder cumulant crossing, and collapsed data from different system sizes onto a single curve using $\beta = 1/8$ and $\nu = 1$. In notebook 06 we built the TFIM and observed its quantum phase transition at $\Gamma_c = J$.

Now we apply the exact same FSS machinery to the quantum transition. The control parameter is $\Gamma/J$ instead of temperature, but every observable — the Binder cumulant, the susceptibility, the data collapse — works identically. The quantum-to-classical mapping guarantees this: the 1D TFIM and the 2D classical Ising model share their universality class and critical exponents.

This notebook completes the connection between the classical and quantum worlds begun in notebook 06.

**What you will do:**
- Compute the quantum order parameter and Binder cumulant from the TFIM ground state
- Locate $\Gamma_c$ from the Binder crossing without any fitting
- Track the energy gap $\Delta(L, \Gamma)$ and verify it closes as $L^{-z}$ at criticality
- Use the dimensionless ratio $L\Delta$ to locate $\Gamma_c$ independently
- Compute the transverse susceptibility and observe its peak scaling
- Collapse all data using $\beta = 1/8$, $\nu = 1$, and $z = 1$

**Prerequisites:** Notebooks 02 (FSS concepts) and 06 (TFIM Hamiltonian).

## 7.1  Setup: TFIM Hamiltonian and Measurement Functions

We reuse `build_tfim` from notebook 06 verbatim. The measurement functions are new: they extract physical observables from the ground state eigenvector.

All operators that appear — $\hat{\sigma}^z_i$, $(\hat{\sigma}^z_{\rm tot})^k$, the gap $E_1 - E_0$ — are straightforward to evaluate given the eigenvectors, because $\hat{\sigma}^z_i$ is diagonal in the computational basis.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from itertools import product

# ── TFIM Hamiltonian (identical to notebook 06) ───────────────────────────────
def build_tfim(N, J=1.0, Gamma=1.0, pbc=False):
    """
    H = -J sum_i sigma_z_i sigma_z_{i+1}  -  Gamma sum_i sigma_x_i
    Basis: bit i = 1 means spin-up (sigma_z eigenvalue +1).
    """
    dim    = 2 ** N
    states = np.arange(dim)
    rows, cols, vals = [], [], []

    n_bonds = N if pbc else N - 1
    for i in range(n_bonds):
        j    = (i + 1) % N
        bi   = (states >> i) & 1
        bj   = (states >> j) & 1
        sz_i = 2 * bi.astype(float) - 1
        sz_j = 2 * bj.astype(float) - 1
        rows.extend(states.tolist())
        cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())

    for i in range(N):
        s_flip = states ^ (1 << i)
        rows.extend(states.tolist())
        cols.extend(s_flip.tolist())
        vals.extend([-Gamma] * dim)

    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)


# ── Helper: sigma_z_total eigenvalue for each basis state ────────────────────
def sz_tot_array(N):
    """Array of sigma_z_total eigenvalues (integers) for each basis state."""
    states = np.arange(2 ** N)
    n_up   = np.array([bin(int(s)).count('1') for s in states])
    return (2 * n_up - N).astype(float)


# ── Order parameter moments ⟨m^k⟩ from the ground state ────────────────────
def op_moments(psi0, N):
    """
    Returns (m2, m4) where m = sigma_z_tot / N.
    For a Z2-symmetric ground state, <m> = 0 exactly.
    m2 = <m^2> is the order parameter proxy: O(1) ordered, O(1/N) disordered.
    """
    m    = sz_tot_array(N) / N
    prob = np.abs(psi0) ** 2
    m2   = float(np.sum(prob * m ** 2))
    m4   = float(np.sum(prob * m ** 4))
    return m2, m4


# ── Binder cumulant ──────────────────────────────────────────────────────────
def binder(m2, m4):
    """U4 = 1 - <m^4> / (3 <m^2>^2)"""
    if m2 < 1e-15:
        return 0.0
    return 1.0 - m4 / (3.0 * m2 ** 2)


# Quick sanity check: at Gamma=0 (perfectly ordered) m2 should be 1, U4 = 2/3
N_test = 8
H_ord  = build_tfim(N_test, J=1.0, Gamma=0.01, pbc=False)
_, evec_test = eigsh(H_ord, k=1, which='SA')
psi_test     = evec_test[:, 0]
m2_t, m4_t   = op_moments(psi_test, N_test)
print(f"Near Gamma=0 (N={N_test}):  m2 = {m2_t:.4f}  (expected ~1)")
print(f"                           U4 = {binder(m2_t, m4_t):.4f}  (expected 2/3 = {2/3:.4f})")

H_dis = build_tfim(N_test, J=1.0, Gamma=5.0, pbc=False)
_, evec_dis  = eigsh(H_dis, k=1, which='SA')
psi_dis      = evec_dis[:, 0]
m2_d, m4_d   = op_moments(psi_dis, N_test)
print(f"Near Gamma>>J (N={N_test}): m2 = {m2_d:.4f}  (expected ~0)")
print(f"                           U4 = {binder(m2_d, m4_d):.4f}  (expected 0)")

## 7.2  Binder Cumulant Crossing

The Binder cumulant:

$$U_4(L, \Gamma) = 1 - \frac{\langle m^4 \rangle}{3\langle m^2 \rangle^2}$$

takes universal limiting values:
- **Ordered phase** ($\Gamma \to 0$): the ground state is the GHZ superposition $(|\uparrow\cdots\uparrow\rangle + |\downarrow\cdots\downarrow\rangle)/\sqrt{2}$, giving $\langle m^2\rangle = 1$ and $\langle m^4\rangle = 1$, so $U_4 \to 2/3$.
- **Disordered phase** ($\Gamma \to \infty$): the ground state is a product state with $\langle m^k\rangle \sim 1/N^{k/2}$, so $U_4 \to 0$.

At $\Gamma_c$, curves for different system sizes cross. The crossing value of $U_4$ at the critical point is a universal number fixed by the universality class.

All moments here are quantum ground state expectation values — no Monte Carlo sampling is involved. The Z2 symmetry of the TFIM with OBC means $\langle m \rangle = 0$ exactly for any finite $N$, so the Binder cumulant formulation is essential.

In [ ]:
J            = 1.0
Gamma_values = np.linspace(0.4, 1.6, 60)
N_binder     = [8, 10, 12, 14, 16]
colors_N     = plt.cm.viridis(np.linspace(0.15, 0.85, len(N_binder)))

binder_data = {}   # binder_data[N] = array of U4 vs Gamma
m2_data     = {}   # m2_data[N]     = array of m2 vs Gamma

for N in N_binder:
    print(f"N = {N} ...", end=' ', flush=True)
    U4_arr, m2_arr = [], []
    for G in Gamma_values:
        H       = build_tfim(N, J=J, Gamma=G, pbc=False)
        _, evec = eigsh(H, k=1, which='SA')
        psi0    = evec[:, 0]
        m2, m4  = op_moments(psi0, N)
        U4_arr.append(binder(m2, m4))
        m2_arr.append(m2)
    binder_data[N] = np.array(U4_arr)
    m2_data[N]     = np.array(m2_arr)
    print("done")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N, col in zip(N_binder, colors_N):
    axes[0].plot(Gamma_values / J, binder_data[N], lw=2, color=col, label=f"$N = {N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].axhline(2/3, color='grey', ls=':', lw=1, label=r'$U_4 = 2/3$ (ordered limit)')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$U_4$', fontsize=13)
axes[0].set_title('Binder cumulant — crossing at $\\Gamma_c$', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

for N, col in zip(N_binder, colors_N):
    axes[1].plot(Gamma_values / J, np.sqrt(m2_data[N]), lw=2, color=col, label=f"$N = {N}$")
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[1].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[1].set_ylabel(r'$\sqrt{\langle m^2 \rangle}$', fontsize=13)
axes[1].set_title('Order parameter proxy', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Locate crossing: find Gamma where U4(N1) - U4(N2) changes sign
N1, N2 = N_binder[-2], N_binder[-1]
diff   = binder_data[N1] - binder_data[N2]
# Interpolate zero crossing
sign_changes = np.where(np.diff(np.sign(diff)))[0]
if len(sign_changes) > 0:
    idx = sign_changes[0]
    G_cross = Gamma_values[idx] - diff[idx] * (Gamma_values[idx+1] - Gamma_values[idx]) / (diff[idx+1] - diff[idx])
    print(f"\nBinder crossing (N={N1} vs N={N2}): Gamma_c / J = {G_cross/J:.4f}  (exact = 1.0000)")

## 7.3  Energy Gap and the $L\Delta$ Crossing

The spectral gap $\Delta(L, \Gamma) = E_1 - E_0$ is a purely quantum observable — it has no analogue in classical Monte Carlo. It carries direct information about the phase:

- **Ordered phase** ($\Gamma < \Gamma_c$): $\Delta \sim e^{-L/\xi}$, exponentially small — the two quasi-degenerate GHZ states.
- **Disordered phase** ($\Gamma > \Gamma_c$): $\Delta \to 2(\Gamma - J)$ as $L \to \infty$ — finite gap.
- **Critical point** ($\Gamma = \Gamma_c$): $\Delta \sim L^{-z}$ with $z = 1$ — power law closing.

The dimensionless combination $L \cdot \Delta(L, \Gamma)$ obeys its own FSS form:

$$L \cdot \Delta(L, \Gamma) = g\!\left((\Gamma - \Gamma_c)\, L^{1/\nu}\right)$$

At $\Gamma_c$, $L \cdot \Delta$ is size-independent, so curves for different $L$ cross at $\Gamma_c$ — another crossing observable, independent of the Binder cumulant.

In [ ]:
Gamma_gap    = np.linspace(0.5, 1.5, 50)
N_gap        = [8, 10, 12, 14, 16]
gap_data     = {}   # gap_data[N] = array of Delta vs Gamma

for N in N_gap:
    print(f"N = {N} ...", end=' ', flush=True)
    gaps = []
    for G in Gamma_gap:
        H     = build_tfim(N, J=J, Gamma=G, pbc=False)
        evals = np.sort(eigsh(H, k=3, which='SA', return_eigenvectors=False))
        gaps.append(evals[1] - evals[0])
    gap_data[N] = np.array(gaps)
    print("done")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gap vs Gamma/J
for N, col in zip(N_gap, colors_N):
    axes[0].plot(Gamma_gap / J, gap_data[N], lw=2, color=col, label=f"$N = {N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$\Delta = E_1 - E_0$', fontsize=13)
axes[0].set_title('Energy gap', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Right: L * Delta — should cross at Gamma_c
for N, col in zip(N_gap, colors_N):
    axes[1].plot(Gamma_gap / J, N * gap_data[N], lw=2, color=col, label=f"$N = {N}$")
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[1].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[1].set_ylabel(r'$L \cdot \Delta$', fontsize=13)
axes[1].set_title(r'$L\,\Delta$ crossing at $\Gamma_c$', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Gap scaling at criticality: Delta(N) ~ N^{-z}, z=1
N_crit_vals = [6, 8, 10, 12, 14, 16, 18, 20]
gaps_crit   = []
for N in N_crit_vals:
    H     = build_tfim(N, J=J, Gamma=J, pbc=False)
    evals = np.sort(eigsh(H, k=2, which='SA', return_eigenvectors=False))
    gaps_crit.append(evals[1] - evals[0])

slope, intercept = np.polyfit(np.log(N_crit_vals), np.log(gaps_crit), 1)
print(f"Gap scaling at Gamma_c: Delta ~ N^{slope:.3f}  (expected -1.000)")

## 7.4  Transverse Susceptibility

The susceptibility measures the response of the $z$-magnetisation to an infinitesimal $z$-field. Using second-order perturbation theory:

$$\chi = \frac{1}{N} \sum_{n \neq 0} \frac{|\langle E_n | \hat{M}^z | E_0 \rangle|^2}{E_n - E_0}$$

where $\hat{M}^z = \sum_i \hat{\sigma}^z_i$. Since $\hat{M}^z$ is diagonal in the computational basis, the matrix element is simply a dot product of the eigenvectors weighted by the $\sigma^z_{\rm tot}$ eigenvalue:

$$\langle E_n | \hat{M}^z | E_0 \rangle = \sum_s \psi^*_{n,s} \cdot (2n^{\uparrow}_s - N) \cdot \psi_{0,s}$$

At the quantum critical point $\chi$ diverges with exponent $\gamma = 7/4$, peaking on a finite system as $\chi_{\rm max} \sim L^{\gamma/\nu} = L^{7/4}$. We compute $\chi$ using full diagonalisation for moderate $N$.

In [ ]:
def susceptibility_full(N, J, Gamma):
    """
    Compute chi = (1/N) sum_{n!=0} |<En|Mz|E0>|^2 / (En - E0)
    using full diagonalisation (practical up to N~16).
    """
    H       = build_tfim(N, J=J, Gamma=Gamma, pbc=False)
    evals, evecs = np.linalg.eigh(H.toarray())
    # Sort by eigenvalue
    idx     = np.argsort(evals)
    evals   = evals[idx]
    evecs   = evecs[:, idx]

    mz_diag = sz_tot_array(N)          # Mz eigenvalue for each basis state
    mz_psi0 = mz_diag * evecs[:, 0]   # Mz |E0> in basis representation

    chi = 0.0
    for n in range(1, len(evals)):
        mel  = np.dot(evecs[:, n], mz_psi0)   # <En|Mz|E0>  (real, since H real)
        chi += mel ** 2 / (evals[n] - evals[0])
    return chi / N


Gamma_chi = np.linspace(0.5, 1.5, 30)
N_chi     = [8, 10, 12, 14]
chi_data  = {}

for N in N_chi:
    print(f"N = {N} ...", end=' ', flush=True)
    chi_data[N] = np.array([susceptibility_full(N, J, G) for G in Gamma_chi])
    print("done")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N, col in zip(N_chi, colors_N):
    axes[0].plot(Gamma_chi / J, chi_data[N], lw=2, color=col, label=f"$N = {N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$\chi$', fontsize=13)
axes[0].set_title('Transverse susceptibility', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Peak scaling: chi_max ~ L^{gamma/nu} = L^{7/4}
chi_peaks = [np.max(chi_data[N]) for N in N_chi]
slope_chi, intercept_chi = np.polyfit(np.log(N_chi), np.log(chi_peaks), 1)

N_fit = np.linspace(min(N_chi) * 0.9, max(N_chi) * 1.1, 50)
axes[1].loglog(N_chi, chi_peaks, 'o', ms=8, color='steelblue', label='$\\chi_{\\rm max}$')
axes[1].loglog(N_fit, np.exp(intercept_chi) * N_fit ** slope_chi, 'k--', lw=1.5,
               label=f'Fit: $\\chi_{{\\rm max}} \\sim N^{{{slope_chi:.2f}}}$')
axes[1].loglog(N_fit, 0.2 * N_fit ** (7/4), 'r:', lw=1.5,
               label=r'Expected: $N^{7/4} = N^{1.75}$')
axes[1].set_xlabel('$N$', fontsize=13)
axes[1].set_ylabel(r'$\chi_{\rm max}$', fontsize=13)
axes[1].set_title(r'Susceptibility peak scaling $\sim N^{\gamma/\nu}$', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nSusceptibility peak exponent: {slope_chi:.3f}  (expected gamma/nu = 7/4 = 1.750)")

## 7.5  Scaling Collapse

Finite-size scaling predicts that all data for different system sizes collapse onto a single scaling function when plotted as:

$$L^{\beta/\nu}\, \sqrt{\langle m^2 \rangle} \quad \text{vs} \quad (\Gamma - \Gamma_c)\, L^{1/\nu}$$

With the 2D classical Ising exponents $\beta = 1/8$, $\nu = 1$ (from the quantum-to-classical mapping), all curves should collapse. A good collapse confirms both the location of $\Gamma_c$ and the universality class.

We use the Binder crossing value $\Gamma_c \approx 1.0$ (exact for OBC in the thermodynamic limit, approached from above for finite systems with OBC). The data collapse is the strongest possible test: it requires the right $\Gamma_c$, the right $\nu$, and the right $\beta$ simultaneously.

In [ ]:
# Use finer Gamma grid for collapse
Gamma_collapse = np.linspace(0.5, 1.5, 80)
N_collapse     = [8, 10, 12, 14, 16]
m2_collapse    = {}   # reuse data if already computed above, else recompute

# Recompute on the fine grid
for N in N_collapse:
    print(f"N = {N} ...", end=' ', flush=True)
    m2_arr = []
    for G in Gamma_collapse:
        H       = build_tfim(N, J=J, Gamma=G, pbc=False)
        _, evec = eigsh(H, k=1, which='SA')
        m2, _   = op_moments(evec[:, 0], N)
        m2_arr.append(m2)
    m2_collapse[N] = np.array(m2_arr)
    print("done")

# Critical exponents (2D classical Ising = 1D TFIM universality class)
beta_exp = 1.0 / 8.0
nu_exp   = 1.0
Gamma_c  = 1.0   # exact

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw data
for N, col in zip(N_collapse, colors_N):
    axes[0].plot(Gamma_collapse / J, np.sqrt(m2_collapse[N]),
                 lw=2, color=col, label=f"$N = {N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2)
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$\sqrt{\langle m^2 \rangle}$', fontsize=13)
axes[0].set_title('Raw order parameter', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Right: collapsed data
for N, col in zip(N_collapse, colors_N):
    x = (Gamma_collapse - Gamma_c) * N ** (1.0 / nu_exp)
    y = N ** (beta_exp / nu_exp) * np.sqrt(m2_collapse[N])
    axes[1].plot(x, y, lw=2, color=col, alpha=0.9, label=f"$N = {N}$")

axes[1].set_xlabel(r'$(\Gamma - \Gamma_c)\, L^{1/\nu}$', fontsize=13)
axes[1].set_ylabel(r'$L^{\beta/\nu}\, \sqrt{\langle m^2 \rangle}$', fontsize=13)
axes[1].set_title(r'Scaling collapse: $\beta=1/8$, $\nu=1$', fontsize=13)
axes[1].set_xlim(-8, 8)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Finite-size scaling collapse for the TFIM quantum phase transition', fontsize=13)
plt.tight_layout()
plt.show()

## 7.6  Classical vs Quantum: Side-by-Side Comparison

The table below summarises how the FSS machinery from notebook 02 maps onto the quantum problem. The mathematical structure is identical; only the physical interpretation changes.

| Observable | Classical 2D Ising (nb02) | Quantum 1D TFIM (nb07) |
|---|---|---|
| Control parameter | Temperature $T$ | Transverse field $\Gamma/J$ |
| Order parameter | $\langle\|m\|\rangle$ (thermal avg) | $\sqrt{\langle m^2\rangle}$ (GS expectation) |
| Binder cumulant | Crosses at $T_c$ | Crosses at $\Gamma_c$ |
| Susceptibility peak | $\chi_{\rm max} \sim L^{\gamma/\nu}$ | Same |
| Spectral gap | Thermal gap always finite | $\Delta \sim L^{-z}$ at QCP |
| $L\Delta$ crossing | Does not apply | Crosses at $\Gamma_c$ |
| Critical exponents | $\beta=1/8$, $\nu=1$, $\gamma=7/4$ | **Same** (universality) |
| Dynamical exponent $z$ | Not applicable | $z = 1$ |

The fact that the exponents are identical is the quantum-to-classical mapping made concrete: the 1D TFIM at $T = 0$ maps exactly onto the 2D classical Ising model, and they are in the same universality class.

The one observable with no classical counterpart is the **spectral gap** and the $L\Delta$ crossing. In the classical problem, measuring the correlation length requires computing two-point functions; here the gap provides a direct, clean probe of the diverging length scale.

In [ ]:
# Summary figure: all four crossing observables on one page
Gamma_summary = np.linspace(0.6, 1.4, 40)
N_summary     = [10, 12, 14, 16]
cols_sum      = plt.cm.plasma(np.linspace(0.15, 0.85, len(N_summary)))

U4_sum   = {N: [] for N in N_summary}
gap_sum  = {N: [] for N in N_summary}

for N in N_summary:
    print(f"N = {N} ...", end=' ', flush=True)
    for G in Gamma_summary:
        H = build_tfim(N, J=J, Gamma=G, pbc=False)
        evals, evec = eigsh(H, k=3, which='SA')
        evals = np.sort(evals)
        psi0  = evec[:, np.argmin(evals)]
        m2, m4 = op_moments(psi0, N)
        U4_sum[N].append(binder(m2, m4))
        gap_sum[N].append(evals[1] - evals[0])
    U4_sum[N]  = np.array(U4_sum[N])
    gap_sum[N] = np.array(gap_sum[N])
    print("done")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N, col in zip(N_summary, cols_sum):
    axes[0].plot(Gamma_summary / J, U4_sum[N], lw=2, color=col, label=f"$N={N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'Binder $U_4$', fontsize=13)
axes[0].set_title('Binder cumulant crossing', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

for N, col in zip(N_summary, cols_sum):
    axes[1].plot(Gamma_summary / J, N * gap_sum[N], lw=2, color=col, label=f"$N={N}$")
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[1].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[1].set_ylabel(r'$L\,\Delta$', fontsize=13)
axes[1].set_title(r'$L\,\Delta$ crossing', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Two independent methods for locating $\\Gamma_c$', fontsize=13)
plt.tight_layout()
plt.show()

## Exercises

1. **Periodic boundary conditions.** Repeat the Binder cumulant calculation with `pbc=True`. PBC restores translation invariance and typically gives sharper crossings. Compare the crossing location and the value of $U_4$ at the crossing between OBC and PBC.

2. **Extract $\nu$ from the crossing shift.** The crossing of $U_4(N_1, \Gamma)$ and $U_4(N_2, \Gamma)$ shifts with system size as $\Gamma_{\rm cross}(L) - \Gamma_c \sim L^{-1/\nu - \omega}$ where $\omega$ is a correction-to-scaling exponent. Plot the crossing location for consecutive pairs $(N, N+2)$ vs $1/N$ and extrapolate to estimate $\Gamma_c$ in the thermodynamic limit.

3. **Alternative collapse using $L\Delta$.** Perform a scaling collapse of the gap data: plot $\Delta \cdot L^z$ vs $(\Gamma - \Gamma_c)\, L^{1/\nu}$ with $z = 1$, $\nu = 1$. A good collapse confirms the dynamical exponent $z = 1$ independently of the magnetic exponents.

4. **Wrong exponents.** Attempt the scaling collapse from section 7.5 using the mean-field exponents $\beta = 1/2$, $\nu = 1/2$. The collapse should fail — the curves will not align. This illustrates why getting the exponents right matters and why the universality class identification is non-trivial.